In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

#### ※ 1つのテーブルで、**各行を一意に識別する役割**を持つものを`キー（Key）`という。  
- キーとして使用する値は`重複してはならない`。

### 学生テーブル

| 学籍番号 | 住民登録番号 | 氏名 |
|---|---|---|
| 050201 | 050025-1254213 | 千尋 |
| 050202 | 990036-2254213 | ハウル |
| 050203 | 070047-2354213 | キキ |

### 履修テーブル

| 学籍番号 | 科目名 |
|---|---|
| 050201 | Python |
| 050202 | SQL |
| 050203 | データベース |

#### ※ 識別子

- `識別子`：各行をを一意に識別する`1つの属性（カラム）`、または`複数の属性（カラム）の組み合わせ`
- `主キー（PK, Primary Key）`：識別子の中から代表として選択されたキー

## # キーの種類

#### 1) **主キー**（Primary Key）

- `各行を一意に識別するキー`
- `一意性`・`最小性`を満たし、`NULL不可`
- 候補キーの中から代表として選択される

例：学生テーブルでは`学籍番号`または`住民登録番号`の中から1つを選び、主キーに設定できる。

#### 2) **候補キー**（Candidate Key）

- `主キーになれるすべてのキー`
- `一意性`・`最小性`を満たし、`NULL不可`

例：学生テーブルでは`学籍番号`と`住民登録番号`が候補キーになる。

#### 3) **代替キー**（Alternate Key）

- `候補キーのうち、主キーに選ばれなかったキー`

例：`学籍番号`を主キーにした場合、`住民登録番号`は代替キーになる。

#### 4) **外部キー**（Foreign Key）

- `他のテーブルの主キーを参照するキー`
- テーブル間を`関連付ける`

例：履修テーブルの`学籍番号`は、学生テーブルの主キー`学籍番号`を参照するため外部キーになる。

#### 5) **スーパーキー**（Super Key）

- `行を一意に識別できるすべてのキー`
- 識別に不要な属性を含む場合もある
- `一意性`は満たすが、`最小性`は必ずしも満たさない

例：学生テーブルでは`学籍番号`、`住民登録番号`、  
`学籍番号 + 住民登録番号`などがスーパーキーになる。

```text
スーパーキー：一意性 ○
    │
    └──► 候補キー：一意性 ○ ＋ 最小性 ○
              ├──► 主キー：代表として選択されたキー（NOT NULL）
              └──► 代替キー：代表として選択されなかった残りのキー
```

#### 6) 複合キー（Composite Key）

- `2つ以上のカラムを組み合わせて構成するキー`
- 例：`学籍番号 + 科目名`

#### ※ すべてのテーブル（エンティティ）は`識別子`を持つ必要があり、識別子がなければ設計が完了していないと考えるのが一般的。

| 区分 | 内容 |
|---|---|
| 識別子のないエンティティ | 原則として不可 |
| 主キー | リレーショナルデータベースでは設定するのが原則 |
| 自然な識別子がない場合 (例1)| `AUTO_INCREMENT`、`UUID`などの`代理キー（Surrogate Key）`を使用 |
| 中間エンティティ (例2)| `複合キー`または`代理キー`を主キーとして使用 |

(例1) 代理キーの使用

ランダムな`身長・体重・年齢`などを収集  
→ 主キーとなる属性を決められない  
→ `AUTO_INCREMENT`で`no`を生成し、代理キーとして使用する。

In [ ]:
%%sql

CREATE TABLE sample (
  no int AUTO_INCREMENT,
  weight int DEFAULT NULL,
  height int DEFAULT NULL,
  age int DEFAULT NULL,
  PRIMARY KEY (no)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;

++
||
++
++

In [12]:
%%sql

INSERT INTO sample (weight, height, age)
VALUES
(60, 170, 25),
(60, 170, 25),
(55, 165, 23);

++
||
++
++

In [13]:
%%sql

SELECT *
FROM sample;

no,weight,height,age
1,60,170,25
2,60,170,25
3,55,165,23


(例2) ：学生と科目の間に`履修`という中間エンティティを作る場合  
- `学籍番号 + 科目ID`を`複合キー`として主キーにする  
- または`履修ID`を追加し、`代理キー`として主キーにする

```text
履修

履修ID | 学籍番号 | 科目ID
1      | 1001     | 101
2      | 1001     | 102
3      | 1002     | 101
```